In [1]:
"""
Sepsis Prediction — LSTM v4
============================
Changes vs v3:
  1. Recall-constrained threshold  → targets ≥70% recall (clinical priority)
  2. Precision-recall curve plot   → saved to results/pr_curve.png
  3. ROC curve plot                → saved to results/roc_curve.png
  4. Confusion matrix plot         → saved to results/confusion_matrix.png
  5. SHAP explainability           → feature importance for the paper
  6. Best model across ALL runs    → saved as lstm_v4_final.pt
  7. Paper-ready metrics table     → printed at the end

Results from v3 (baseline for comparison):
  AUROC  : 0.8638
  AUPRC  : 0.5319
  F1     : 0.588
  Recall : 0.492   ← too low, fixing with recall-constrained threshold
"""

import os
import csv
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # no display needed
import matplotlib.pyplot as plt
from datetime import datetime

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  ← only section you need to edit
# ─────────────────────────────────────────────────────────────────────────────
PARENT_DIR    = ".."
HEADER_FOLDER = os.path.join(PARENT_DIR, "processed_training_Ay")

ALL_FOLDERS = [
    os.path.join(PARENT_DIR, "processed_training"),
    os.path.join(PARENT_DIR, "processed_training_setB"),
    os.path.join(PARENT_DIR, "processed_training_Ay"),
]

SAVE_DIR      = "saved_models"
RESULTS_DIR   = "results"
MODEL_VERSION = "lstm_v4"

EARLY_HOURS   = 6     # predict sepsis this many hours before onset
MAX_TIMESTEPS = 48    # last 48 hours only
EPOCHS        = 50
BATCH_SIZE    = 64
PATIENCE      = 10
VAL_SIZE      = 0.2
RANDOM_STATE  = 42
MIN_RECALL    = 0.70  # clinical target — catch at least 70% of sepsis cases

os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Device
# ─────────────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Feature columns
# ─────────────────────────────────────────────────────────────────────────────
all_columns = set()
if not os.path.exists(HEADER_FOLDER):
    raise FileNotFoundError(f"Header folder not found: {os.path.abspath(HEADER_FOLDER)}")

for fname in os.listdir(HEADER_FOLDER):
    if fname.endswith(".psv"):
        with open(os.path.join(HEADER_FOLDER, fname), "r") as fh:
            header = fh.readline().strip()
            if header:
                all_columns.update(header.split("|"))

FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")
N_FEATURES = len(FEATURE_COLUMNS)
print(f"Feature count: {N_FEATURES}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Load all patients
# ─────────────────────────────────────────────────────────────────────────────
def load_all_patients(folders, feature_columns, early_hours=6):
    X, y = [], []
    seen_files = set()

    for folder in folders:
        if not os.path.exists(folder):
            print(f"  ⚠ Skipping missing folder: {folder}")
            continue
        for fname in sorted(os.listdir(folder)):
            if not fname.endswith(".psv"):
                continue
            if fname in seen_files:
                continue
            seen_files.add(fname)

            df = pd.read_csv(os.path.join(folder, fname), sep="|")
            df = df.reindex(columns=feature_columns + ["SepsisLabel"])
            df[feature_columns] = df[feature_columns].fillna(0.0)

            features = df[feature_columns].values.astype(np.float32)
            labels   = df["SepsisLabel"].values

            if labels.max() == 1:
                t_sepsis = np.where(labels == 1)[0][0]
                cutoff   = t_sepsis - early_hours
                if cutoff <= 0:
                    continue
                X.append(features[:cutoff])
                y.append(1)
            else:
                X.append(features)
                y.append(0)

    return X, np.array(y, dtype=np.int64)


print("\nLoading all patient data...")
X_all, y_all = load_all_patients(ALL_FOLDERS, FEATURE_COLUMNS, EARLY_HOURS)
print(f"Total patients : {len(y_all)}")
print(f"Sepsis (1)     : {y_all.sum()}  ({y_all.mean():.3f})")
print(f"No sepsis (0)  : {(y_all == 0).sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Stratified split
# ─────────────────────────────────────────────────────────────────────────────
indices = np.arange(len(y_all))
idx_train, idx_val = train_test_split(
    indices,
    test_size    = VAL_SIZE,
    stratify     = y_all,
    random_state = RANDOM_STATE,
)

X_train_raw = [X_all[i] for i in idx_train]
X_val_raw   = [X_all[i] for i in idx_val]
y_train     = y_all[idx_train]
y_val       = y_all[idx_val]

print(f"\nAfter stratified split:")
print(f"  Train : {len(y_train)} patients | sepsis rate: {y_train.mean():.4f}")
print(f"  Val   : {len(y_val)}  patients | sepsis rate: {y_val.mean():.4f}")
if abs(y_train.mean() - y_val.mean()) > 0.005:
    print("⚠ WARNING: sepsis rates differ — check stratify= argument")
else:
    print("✓ Sepsis rates match")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Pad sequences
# ─────────────────────────────────────────────────────────────────────────────
def pad_sequences(X, max_len):
    X_pad = np.zeros((len(X), max_len, N_FEATURES), dtype=np.float32)
    for i, seq in enumerate(X):
        L = len(seq)
        if L >= max_len:
            X_pad[i] = seq[-max_len:]
        else:
            X_pad[i, max_len - L:] = seq
    return X_pad

X_train = pad_sequences(X_train_raw, MAX_TIMESTEPS)
X_val   = pad_sequences(X_val_raw,   MAX_TIMESTEPS)
print(f"\nX_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Normalise features
# ─────────────────────────────────────────────────────────────────────────────
n_train, T, F = X_train.shape
n_val         = X_val.shape[0]

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, F)).reshape(n_train, T, F).astype(np.float32)
X_val   = scaler.transform(X_val.reshape(-1, F)).reshape(n_val, T, F).astype(np.float32)

scaler_path = os.path.join(SAVE_DIR, f"{MODEL_VERSION}_scaler.pkl")
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved → {scaler_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Class imbalance weights
# ─────────────────────────────────────────────────────────────────────────────
class_counts    = np.bincount(y_train)
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\nClass counts — 0: {class_counts[0]}, 1: {class_counts[1]}")
print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")

sample_weights = np.where(y_train == 1,
                          1.0 / class_counts[1],
                          1.0 / class_counts[0])
pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32).to(device)

# ─────────────────────────────────────────────────────────────────────────────
# 8. DataLoaders
# ─────────────────────────────────────────────────────────────────────────────
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val, dtype=torch.float32),
)

sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float32),
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 9. Model — Bidirectional LSTM + Temporal Attention
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,
        )
        lstm_out_dim = hidden_dim * 2

        self.attention  = nn.Linear(lstm_out_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        attn_scores  = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context)

# ─────────────────────────────────────────────────────────────────────────────
# 10. Experiment logger
# ─────────────────────────────────────────────────────────────────────────────
LOG_PATH   = os.path.join(RESULTS_DIR, "model_log.csv")
LOG_FIELDS = ["timestamp", "version", "params", "epochs_run",
              "auroc", "auprc", "f1_sepsis", "recall_sepsis",
              "precision_sepsis", "threshold", "notes"]

def log_experiment(version, params, epochs_run, auroc, auprc,
                   f1, recall, precision, threshold, notes=""):
    row = {
        "timestamp":        datetime.now().strftime("%Y-%m-%d %H:%M"),
        "version":          version,
        "params":           str(params),
        "epochs_run":       epochs_run,
        "auroc":            round(auroc, 4),
        "auprc":            round(auprc, 4),
        "f1_sepsis":        round(f1, 4),
        "recall_sepsis":    round(recall, 4),
        "precision_sepsis": round(precision, 4),
        "threshold":        round(threshold, 3),
        "notes":            notes,
    }
    write_header = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  → Logged to {LOG_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 11. Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def get_val_probs(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            logits  = model(batch_x).squeeze(1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)


def best_threshold_by_recall(y_true, y_prob, min_recall=0.70):
    """
    Find the highest-precision threshold that still achieves min_recall.
    Clinically: missing sepsis (false negative) is worse than a false alarm.
    Falls back to F1-optimal threshold if min_recall cannot be achieved.
    """
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_prob)

    # thresholds array is 1 shorter than precision/recall arrays
    valid = recall_arr[:-1] >= min_recall
    if valid.any():
        # Among valid thresholds, pick the one with highest precision
        best_idx = np.where(valid, precision_arr[:-1], 0).argmax()
        return (thresholds[best_idx],
                recall_arr[best_idx],
                precision_arr[best_idx])
    else:
        print(f"  ⚠ Could not achieve recall ≥ {min_recall:.0%} — using F1-optimal threshold")
        denom  = precision_arr[:-1] + recall_arr[:-1]
        denom  = np.where(denom == 0, 1e-8, denom)
        f1_arr = 2 * (precision_arr[:-1] * recall_arr[:-1]) / denom
        best_idx = np.argmax(f1_arr)
        return thresholds[best_idx], recall_arr[best_idx], precision_arr[best_idx]

# ─────────────────────────────────────────────────────────────────────────────
# 12. Plot helpers  (all saved to results/)
# ─────────────────────────────────────────────────────────────────────────────
def plot_roc_curve(y_true, y_prob, auroc):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="#1A73E8", lw=2, label=f"LSTM (AUC = {auroc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.50)")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Sepsis Prediction")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    path = os.path.join(RESULTS_DIR, "roc_curve.png")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  → ROC curve saved to {path}")


def plot_pr_curve(y_true, y_prob, auprc, threshold, recall_at_t, precision_at_t):
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall_arr, precision_arr, color="#E8431A", lw=2,
            label=f"LSTM (AUPRC = {auprc:.4f})")
    ax.axvline(x=recall_at_t, color="gray", linestyle="--", lw=1, alpha=0.7)
    ax.scatter([recall_at_t], [precision_at_t], color="#E8431A", zorder=5,
               label=f"Threshold={threshold:.3f}\nRecall={recall_at_t:.2f}, Prec={precision_at_t:.2f}")
    baseline = y_true.mean()
    ax.axhline(y=baseline, color="navy", linestyle=":", lw=1,
               label=f"Baseline (prevalence = {baseline:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — Sepsis Prediction")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    path = os.path.join(RESULTS_DIR, "pr_curve.png")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  → PR curve saved to {path}")


def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["No Sepsis", "Sepsis"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("Confusion Matrix — Sepsis Prediction")
    path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  → Confusion matrix saved to {path}")


def plot_training_history(history, tag):
    epochs_x = [h["epoch"] for h in history]
    aurocs    = [h["auroc"] for h in history]
    losses    = [h["loss"]  for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(epochs_x, aurocs, color="#1A73E8", lw=2)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Validation AUROC")
    ax1.set_title("AUROC over training"); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_x, losses, color="#E8431A", lw=2)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Training Loss")
    ax2.set_title("Loss over training"); ax2.grid(True, alpha=0.3)

    path = os.path.join(RESULTS_DIR, f"training_history_{tag}.png")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  → Training history saved to {path}")

# ─────────────────────────────────────────────────────────────────────────────
# 13. Training loop
# ─────────────────────────────────────────────────────────────────────────────
def train_model(params, tag):
    print(f"\n{'='*55}")
    print(f"Training: {params}")
    print(f"{'='*55}")

    model = SepsisLSTM(
        input_dim  = N_FEATURES,
        hidden_dim = params["hidden_dim"],
        num_layers = params["num_layers"],
        dropout    = params["dropout"],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=params["lr"], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5
    )

    best_auroc       = 0.0
    best_state       = None
    patience_counter = 0
    history          = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x).squeeze(1)
            loss   = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        y_prob_val = get_val_probs(model, val_loader)
        auroc      = roc_auc_score(y_val, y_prob_val)
        avg_loss   = total_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]

        scheduler.step(auroc)
        history.append({"epoch": epoch, "loss": avg_loss, "auroc": auroc})

        marker = ""
        if auroc > best_auroc:
            best_auroc       = auroc
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker           = " ✓ best"
            torch.save(best_state, os.path.join(SAVE_DIR, f"{tag}_best.pt"))
        else:
            patience_counter += 1

        print(f"  Epoch {epoch:02d}/{EPOCHS} | loss: {avg_loss:.4f} | "
              f"AUROC: {auroc:.4f} | lr: {current_lr:.2e}{marker}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, best_auroc, history

# ─────────────────────────────────────────────────────────────────────────────
# 14. Grid search
#     Best params from v3: dropout=0.4, hidden_dim=64, lr=1e-4, num_layers=2
#     Grid now focuses around that winner + a few larger variants
# ─────────────────────────────────────────────────────────────────────────────
param_grid = ParameterGrid({
    "hidden_dim": [64, 128],
    "num_layers": [2],
    "dropout":    [0.3, 0.4],
    "lr":         [1e-4, 3e-4],
})

best_overall_auroc   = 0.0
best_overall_model   = None
best_overall_params  = None
best_overall_history = []

for i, params in enumerate(param_grid):
    tag = f"{MODEL_VERSION}_run{i+1}"
    model, auroc, history = train_model(params, tag)
    if auroc > best_overall_auroc:
        best_overall_auroc   = auroc
        best_overall_model   = model
        best_overall_params  = params
        best_overall_history = history

print(f"\n{'='*55}")
print(f"Best grid search AUROC : {best_overall_auroc:.4f}")
print(f"Best params            : {best_overall_params}")

# ─────────────────────────────────────────────────────────────────────────────
# 15. Final evaluation with recall-constrained threshold
# ─────────────────────────────────────────────────────────────────────────────
y_prob = get_val_probs(best_overall_model, val_loader)
auroc  = roc_auc_score(y_val, y_prob)
auprc  = average_precision_score(y_val, y_prob)

best_t, achieved_recall, achieved_precision = best_threshold_by_recall(
    y_val, y_prob, min_recall=MIN_RECALL
)
y_pred = (y_prob > best_t).astype(int)

report         = classification_report(y_val, y_pred, output_dict=True)
sepsis_key     = "1" if "1" in report else "1.0"
sepsis_metrics = report.get(sepsis_key, {})

print(f"\n{'='*55}")
print("FINAL PERFORMANCE REPORT")
print(f"{'='*55}")
print(f"ROC-AUC   : {auroc:.4f}   (target ≥ 0.80)")
print(f"AUPRC     : {auprc:.4f}   (target ≥ 0.30)")
print(f"Threshold : {best_t:.4f}  (recall-constrained, target ≥ {MIN_RECALL:.0%})")
print(f"Recall    : {achieved_recall:.4f}  ← sepsis patients correctly flagged")
print(f"Precision : {achieved_precision:.4f}  ← of all alerts, how many are real sepsis")
print()
print(classification_report(y_val, y_pred, target_names=["No Sepsis", "Sepsis"]))

# ─────────────────────────────────────────────────────────────────────────────
# 16. Save plots
# ─────────────────────────────────────────────────────────────────────────────
print("\nSaving plots...")
plot_roc_curve(y_val, y_prob, auroc)
plot_pr_curve(y_val, y_prob, auprc, best_t, achieved_recall, achieved_precision)
plot_confusion_matrix(y_val, y_pred)
plot_training_history(best_overall_history, "best")

# ─────────────────────────────────────────────────────────────────────────────
# 17. Save model + log
# ─────────────────────────────────────────────────────────────────────────────
final_path = os.path.join(SAVE_DIR, f"{MODEL_VERSION}_final.pt")
torch.save(best_overall_model.state_dict(), final_path)
print(f"\nModel saved → {final_path}")

log_experiment(
    version    = MODEL_VERSION,
    params     = best_overall_params,
    epochs_run = len(best_overall_history),
    auroc      = auroc,
    auprc      = auprc,
    f1         = sepsis_metrics.get("f1-score",  0),
    recall     = sepsis_metrics.get("recall",    0),
    precision  = sepsis_metrics.get("precision", 0),
    threshold  = best_t,
    notes      = (f"Recall-constrained threshold (≥{MIN_RECALL:.0%}), "
                  f"bidirectional+attention, StandardScaler, 48-step window"),
)

# ─────────────────────────────────────────────────────────────────────────────
# 18. SHAP feature importance
#     Uses a random sample of 200 val patients for speed
# ─────────────────────────────────────────────────────────────────────────────
print("\nComputing SHAP feature importance...")
try:
    import shap

    # SHAP works on a CPU model with a small background set
    shap_model = best_overall_model.to("cpu")
    shap_model.eval()

    def model_predict(x_np):
        with torch.no_grad():
            t = torch.tensor(x_np, dtype=torch.float32)
            logits = shap_model(t).squeeze(1)
            return torch.sigmoid(logits).numpy()

    # Use 50 background samples, explain 200 val samples
    rng         = np.random.default_rng(42)
    bg_idx      = rng.choice(len(X_val), size=50,  replace=False)
    exp_idx     = rng.choice(len(X_val), size=200, replace=False)
    background  = X_val[bg_idx]
    to_explain  = X_val[exp_idx]

    explainer   = shap.KernelExplainer(model_predict, background.reshape(50, -1))
    shap_values = explainer.shap_values(to_explain.reshape(200, -1), nsamples=100)

    # Average absolute SHAP across timesteps to get per-feature importance
    shap_per_feature = np.abs(shap_values).reshape(200, MAX_TIMESTEPS, N_FEATURES).mean(axis=(0, 1))
    feature_importance = pd.DataFrame({
        "feature":    FEATURE_COLUMNS,
        "shap_value": shap_per_feature,
    }).sort_values("shap_value", ascending=False)

    print("\nTop 15 most important features (SHAP):")
    print(feature_importance.head(15).to_string(index=False))

    # Save feature importance CSV
    fi_path = os.path.join(RESULTS_DIR, "feature_importance_shap.csv")
    feature_importance.to_csv(fi_path, index=False)
    print(f"  → Feature importance saved to {fi_path}")

    # Bar plot of top 15
    fig, ax = plt.subplots(figsize=(8, 6))
    top15 = feature_importance.head(15)
    ax.barh(top15["feature"][::-1], top15["shap_value"][::-1], color="#1A73E8")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("Top 15 features — SHAP importance")
    ax.grid(True, alpha=0.3, axis="x")
    shap_plot_path = os.path.join(RESULTS_DIR, "shap_feature_importance.png")
    fig.tight_layout()
    fig.savefig(shap_plot_path, dpi=150)
    plt.close(fig)
    print(f"  → SHAP plot saved to {shap_plot_path}")

    # Move model back to original device
    best_overall_model.to(device)

except ImportError:
    print("  ⚠ SHAP not installed. Run: pip install shap")
    print("    Then re-run this section for feature importance.")
except Exception as e:
    print(f"  ⚠ SHAP failed: {e}")
    best_overall_model.to(device)

# ─────────────────────────────────────────────────────────────────────────────
# 19. Architecture summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("Model architecture:")
print(f"{'='*55}")
print(best_overall_model)

print(f"\n{'='*55}")
print("Layer parameter shapes:")
print(f"{'='*55}")
for name, param in best_overall_model.named_parameters():
    if param.requires_grad:
        print(f"  {name:40} {tuple(param.shape)}")

# ─────────────────────────────────────────────────────────────────────────────
# 20. Paper-ready metrics table
# ─────────────────────────────────────────────────────────────────────────────
f1_val       = sepsis_metrics.get("f1-score",  0)
recall_val   = sepsis_metrics.get("recall",    0)
precision_val= sepsis_metrics.get("precision", 0)

print(f"\n{'='*55}")
print("PAPER-READY METRICS TABLE")
print(f"{'='*55}")
print(f"  {'Metric':<25} {'v1':>8} {'v2':>8} {'v3':>8} {'v4':>8}")
print(f"  {'-'*57}")
print(f"  {'AUROC':<25} {'0.617':>8} {'0.689':>8} {'0.864':>8} {auroc:>8.3f}")
print(f"  {'AUPRC':<25} {'—':>8} {'—':>8} {'0.532':>8} {auprc:>8.3f}")
print(f"  {'F1 (sepsis)':<25} {'0.140':>8} {'0.335':>8} {'0.588':>8} {f1_val:>8.3f}")
print(f"  {'Recall (sepsis)':<25} {'0.270':>8} {'0.287':>8} {'0.492':>8} {recall_val:>8.3f}")
print(f"  {'Precision (sepsis)':<25} {'0.080':>8} {'0.190':>8} {'0.730':>8} {precision_val:>8.3f}")
print(f"  {'Threshold':<25} {'0.900':>8} {'0.912':>8} {'0.997':>8} {best_t:>8.3f}")
print(f"  {'Dataset':<25} {'PhysioNet':>8} {'PhysioNet':>8} {'AMS-UMC':>8} {'AMS-UMC':>8}")
print()
print("  Files saved for paper:")
print(f"  results/roc_curve.png")
print(f"  results/pr_curve.png")
print(f"  results/confusion_matrix.png")
print(f"  results/shap_feature_importance.png")
print(f"  results/feature_importance_shap.csv")
print(f"  results/model_log.csv")

Using device: cuda
Feature count: 32

Loading all patient data...
Total patients : 39558
Sepsis (1)     : 2154  (0.054)
No sepsis (0)  : 37404

After stratified split:
  Train : 31646 patients | sepsis rate: 0.0544
  Val   : 7912  patients | sepsis rate: 0.0545
✓ Sepsis rates match

X_train : (31646, 48, 32)
X_val   : (7912, 48, 32)

Scaler saved → saved_models/lstm_v4_scaler.pkl

Class counts — 0: 29923, 1: 1723
Imbalance ratio : 17.4:1

Training: {'dropout': 0.3, 'hidden_dim': 64, 'lr': 0.0001, 'num_layers': 2}
  Epoch 01/50 | loss: 2.3158 | AUROC: 0.8203 | lr: 1.00e-04 ✓ best
  Epoch 02/50 | loss: 1.4624 | AUROC: 0.8350 | lr: 1.00e-04 ✓ best
  Epoch 03/50 | loss: 1.4250 | AUROC: 0.8403 | lr: 1.00e-04 ✓ best
  Epoch 04/50 | loss: 1.3561 | AUROC: 0.8430 | lr: 1.00e-04 ✓ best
  Epoch 05/50 | loss: 1.3302 | AUROC: 0.8450 | lr: 1.00e-04 ✓ best
  Epoch 06/50 | loss: 1.2847 | AUROC: 0.8447 | lr: 1.00e-04
  Epoch 07/50 | loss: 1.2539 | AUROC: 0.8505 | lr: 1.00e-04 ✓ best
  Epoch 08/50 | los

In [2]:
"""
Sepsis Prediction — SHAP Explainability
=========================================
Run this AFTER sepsis_lstm_v4.py has finished training.
Requires:
  - saved_models/lstm_v4_final.pt
  - saved_models/lstm_v4_scaler.pkl
  - X_val, y_val, FEATURE_COLUMNS already in memory
    (run this in the same notebook/session as v4, or re-load below)

Produces (all saved to results/):
  shap_bar_top15.png         ← top 15 features by mean |SHAP|   (paper figure)
  shap_beeswarm.png          ← beeswarm plot showing direction   (paper figure)
  shap_temporal_heatmap.png  ← which hours matter most           (paper figure)
  shap_sepsis_vs_nonsepsis.png ← SHAP comparison by class
  feature_importance_shap.csv  ← full ranked table               (paper table)
  shap_values.npy              ← raw values saved for reproducibility
"""

import os
import shap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import joblib

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — match exactly what you used in v4
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH    = "saved_models/lstm_v4_final.pt"
SCALER_PATH   = "saved_models/lstm_v4_scaler.pkl"
RESULTS_DIR   = "results"
N_FEATURES    = 32
MAX_TIMESTEPS = 48
HIDDEN_DIM    = 64      # best params from v4
NUM_LAYERS    = 2
DROPOUT       = 0.4

N_BACKGROUND  = 100    # background samples for KernelExplainer
N_EXPLAIN     = 300    # patients to explain (more = slower but richer plots)
N_EXPLAIN_SEP = 100    # of N_EXPLAIN, how many to pick from sepsis class
RANDOM_SEED   = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Feature columns — must match training order
#    If running standalone, re-load from your header folder
# ─────────────────────────────────────────────────────────────────────────────
# If already in memory from v4 session, skip this block.
# Otherwise uncomment and set HEADER_FOLDER:

# HEADER_FOLDER = "../processed_training_Ay"
# all_columns = set()
# for fname in os.listdir(HEADER_FOLDER):
#     if fname.endswith(".psv"):
#         with open(os.path.join(HEADER_FOLDER, fname)) as fh:
#             hdr = fh.readline().strip()
#             if hdr: all_columns.update(hdr.split("|"))
# FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Model definition (must match v4 exactly)
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,
        )
        lstm_out_dim = hidden_dim * 2
        self.attention  = nn.Linear(lstm_out_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        attn_scores  = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context)


# ─────────────────────────────────────────────────────────────────────────────
# 3. Load model (CPU — SHAP runs on CPU)
# ─────────────────────────────────────────────────────────────────────────────
model = SepsisLSTM(
    input_dim  = N_FEATURES,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()
print(f"✓ Model loaded from {MODEL_PATH}")

scaler = joblib.load(SCALER_PATH)
print(f"✓ Scaler loaded from {SCALER_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Prediction wrapper for SHAP
#    Input:  2D array (n_samples, timesteps * features)  ← SHAP flattens input
#    Output: 1D array of sepsis probabilities
# ─────────────────────────────────────────────────────────────────────────────
def predict_proba(x_flat):
    """Reshape flat input back to (batch, timesteps, features), run model."""
    x_3d = x_flat.reshape(-1, MAX_TIMESTEPS, N_FEATURES)
    t    = torch.tensor(x_3d, dtype=torch.float32)
    with torch.no_grad():
        logits = model(t).squeeze(1)
        probs  = torch.sigmoid(logits).numpy()
    return probs


# ─────────────────────────────────────────────────────────────────────────────
# 5. Sample patients for background and explanation
#    Strategy: balanced sample — half sepsis, half non-sepsis for explanation
#    Background: random sample of non-sepsis (represents "typical" patient)
# ─────────────────────────────────────────────────────────────────────────────
# X_val and y_val must be available (from v4 session or re-loaded)
# They should already be scaled (StandardScaler applied in v4)

print(f"\nX_val shape : {X_val.shape}")
print(f"y_val sepsis: {y_val.sum()} / {len(y_val)}")

sepsis_idx    = np.where(y_val == 1)[0]
nonsepsis_idx = np.where(y_val == 0)[0]

# Background: random non-sepsis patients (what the model considers "normal")
bg_idx = rng.choice(nonsepsis_idx, size=N_BACKGROUND, replace=False)
background = X_val[bg_idx].reshape(N_BACKGROUND, -1)   # flatten to 2D

# Explanation set: balanced — sepsis + non-sepsis
n_sep  = min(N_EXPLAIN_SEP, len(sepsis_idx))
n_non  = N_EXPLAIN - n_sep
exp_sep_idx = rng.choice(sepsis_idx,    size=n_sep, replace=False)
exp_non_idx = rng.choice(nonsepsis_idx, size=n_non, replace=False)
exp_idx     = np.concatenate([exp_sep_idx, exp_non_idx])
exp_labels  = np.concatenate([np.ones(n_sep), np.zeros(n_non)]).astype(int)

to_explain  = X_val[exp_idx].reshape(len(exp_idx), -1)  # flatten to 2D
print(f"\nBackground : {N_BACKGROUND} non-sepsis patients")
print(f"Explaining : {len(exp_idx)} patients ({n_sep} sepsis + {n_non} non-sepsis)")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Run SHAP KernelExplainer
#    This is the slow step — ~5-15 mins depending on machine
#    nsamples=200 is a good balance of speed vs accuracy
# ─────────────────────────────────────────────────────────────────────────────
print("\nInitialising KernelExplainer (this takes a moment)...")
explainer = shap.KernelExplainer(predict_proba, background)

print(f"Computing SHAP values for {len(exp_idx)} patients...")
print("(This will take ~5-15 minutes — progress shown below)\n")
shap_values_flat = explainer.shap_values(to_explain, nsamples=200)
# shap_values_flat shape: (n_explain, timesteps * features)

# Reshape back to 3D: (n_explain, timesteps, features)
shap_3d = shap_values_flat.reshape(len(exp_idx), MAX_TIMESTEPS, N_FEATURES)
print(f"\n✓ SHAP values computed. Shape: {shap_3d.shape}")

# Save raw values for reproducibility
np.save(os.path.join(RESULTS_DIR, "shap_values.npy"), shap_3d)
np.save(os.path.join(RESULTS_DIR, "shap_exp_labels.npy"), exp_labels)
print(f"✓ Raw SHAP values saved → results/shap_values.npy")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Feature importance — mean |SHAP| across all timesteps and patients
# ─────────────────────────────────────────────────────────────────────────────
# Shape: (n_features,)
shap_per_feature = np.abs(shap_3d).mean(axis=(0, 1))

feature_importance = pd.DataFrame({
    "feature":    FEATURE_COLUMNS,
    "shap_value": shap_per_feature,
    "rank":       pd.Series(shap_per_feature).rank(ascending=False).astype(int),
}).sort_values("shap_value", ascending=False).reset_index(drop=True)

print("\n" + "="*45)
print("TOP 15 FEATURES BY SHAP IMPORTANCE")
print("="*45)
print(feature_importance.head(15).to_string(index=False))

fi_path = os.path.join(RESULTS_DIR, "feature_importance_shap.csv")
feature_importance.to_csv(fi_path, index=False)
print(f"\n✓ Full table saved → {fi_path}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — Horizontal bar chart: top 15 features (paper figure)
# ─────────────────────────────────────────────────────────────────────────────
top15 = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors  = ["#1A73E8" if i < 5 else "#5BA4F5" if i < 10 else "#A8C8FA"
           for i in range(15)]
bars = ax.barh(
    top15["feature"].values[::-1],
    top15["shap_value"].values[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.5
)
ax.set_xlabel("Mean |SHAP value| (impact on sepsis probability)", fontsize=11)
ax.set_title("Feature Importance — SHAP Analysis\nSepsis Prediction LSTM (AmsterdamUMCdb)",
             fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
ax.spines[["top", "right"]].set_visible(False)

# Annotate bar values
for bar, val in zip(bars, top15["shap_value"].values[::-1]):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8, color="#333333")

fig.tight_layout()
path1 = os.path.join(RESULTS_DIR, "shap_bar_top15.png")
fig.savefig(path1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Bar chart saved → {path1}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — Beeswarm plot using SHAP's built-in (shows direction of effect)
#           Red = high feature value pushes toward sepsis prediction
#           Blue = high feature value pushes away from sepsis prediction
# ─────────────────────────────────────────────────────────────────────────────

# For beeswarm we need mean SHAP per patient per feature (collapse timesteps)
shap_per_patient_feature = shap_3d.mean(axis=1)   # (n_explain, n_features)
X_mean_per_patient       = to_explain.reshape(
    len(exp_idx), MAX_TIMESTEPS, N_FEATURES
).mean(axis=1)  # mean feature values per patient, (n_explain, n_features)

shap_explanation = shap.Explanation(
    values          = shap_per_patient_feature,
    base_values     = np.full(len(exp_idx), explainer.expected_value),
    data            = X_mean_per_patient,
    feature_names   = FEATURE_COLUMNS,
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_explanation, max_display=15, show=False)
plt.title("SHAP Beeswarm — Feature Direction of Effect\n"
          "Red = high value → higher sepsis risk | Blue = high value → lower risk",
          fontsize=11, fontweight="bold")
plt.tight_layout()
path2 = os.path.join(RESULTS_DIR, "shap_beeswarm.png")
plt.savefig(path2, dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Beeswarm plot saved → {path2}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — Temporal heatmap: which HOURS matter most for prediction
#           Rows = top 10 features, Columns = hour 0..47
#           Colour = mean |SHAP| at that timestep
# ─────────────────────────────────────────────────────────────────────────────
top10_features = feature_importance.head(10)["feature"].tolist()
top10_idx      = [FEATURE_COLUMNS.index(f) for f in top10_features]

# Mean |SHAP| per feature per timestep across all explained patients
# shap_3d: (n_explain, timesteps, features)
temporal_importance = np.abs(shap_3d).mean(axis=0)           # (timesteps, features)
temporal_top10      = temporal_importance[:, top10_idx].T    # (10, timesteps)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(temporal_top10, aspect="auto", cmap="YlOrRd", interpolation="nearest")

ax.set_yticks(range(len(top10_features)))
ax.set_yticklabels(top10_features, fontsize=9)
ax.set_xlabel("Hour (0 = oldest, 47 = most recent before sepsis onset)", fontsize=10)
ax.set_title("Temporal SHAP Heatmap — When Do Features Matter?\n"
             "Darker = higher impact on sepsis prediction at that hour",
             fontsize=11, fontweight="bold")

# X-axis: label every 6 hours
xtick_positions = list(range(0, MAX_TIMESTEPS, 6))
ax.set_xticks(xtick_positions)
ax.set_xticklabels([f"h-{MAX_TIMESTEPS-x}" for x in xtick_positions], fontsize=8)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean |SHAP value|", fontsize=9)

fig.tight_layout()
path3 = os.path.join(RESULTS_DIR, "shap_temporal_heatmap.png")
fig.savefig(path3, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Temporal heatmap saved → {path3}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 — Sepsis vs Non-Sepsis SHAP comparison
#           Shows which features differ most between the two groups
# ─────────────────────────────────────────────────────────────────────────────
shap_sepsis    = np.abs(shap_3d[exp_labels == 1]).mean(axis=(0, 1))  # (n_features,)
shap_nonsepsis = np.abs(shap_3d[exp_labels == 0]).mean(axis=(0, 1))

# Top 12 by combined importance
combined = (shap_sepsis + shap_nonsepsis) / 2
top12_idx_combined = np.argsort(combined)[::-1][:12]
top12_names        = [FEATURE_COLUMNS[i] for i in top12_idx_combined]

x      = np.arange(len(top12_names))
width  = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, shap_sepsis[top12_idx_combined],    width, label="Sepsis patients",
       color="#E8431A", alpha=0.85)
ax.bar(x + width/2, shap_nonsepsis[top12_idx_combined], width, label="Non-sepsis patients",
       color="#1A73E8", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(top12_names, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Mean |SHAP value|", fontsize=10)
ax.set_title("Feature Importance: Sepsis vs Non-Sepsis Patients\n"
             "Shows which features the model uses differently for each group",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
path4 = os.path.join(RESULTS_DIR, "shap_sepsis_vs_nonsepsis.png")
fig.savefig(path4, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Sepsis vs non-sepsis plot saved → {path4}")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Final summary printout
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("SHAP ANALYSIS COMPLETE")
print(f"{'='*55}")
print(f"\nTop 10 features driving sepsis prediction:")
for i, row in feature_importance.head(10).iterrows():
    print(f"  {i+1:2}. {row['feature']:<20}  SHAP: {row['shap_value']:.5f}")

print(f"\nFiles saved for paper:")
print(f"  results/shap_bar_top15.png          ← Figure: feature importance bar chart")
print(f"  results/shap_beeswarm.png           ← Figure: direction of feature effects")
print(f"  results/shap_temporal_heatmap.png   ← Figure: which hours matter most")
print(f"  results/shap_sepsis_vs_nonsepsis.png← Figure: sepsis vs non-sepsis comparison")
print(f"  results/feature_importance_shap.csv ← Table:  full ranked feature list")
print(f"  results/shap_values.npy             ← Data:   raw values for reproducibility")

print(f"\nFor your Methods section:")
print(f"  'SHAP (SHapley Additive exPlanations) KernelExplainer was applied")
print(f"  to {len(exp_idx)} validation patients ({n_sep} sepsis, {n_non} non-sepsis)")
print(f"  using {N_BACKGROUND} non-sepsis background samples to explain")
print(f"  individual predictions and identify the most influential clinical features.'")

/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Model loaded from saved_models/lstm_v4_final.pt
✓ Scaler loaded from saved_models/lstm_v4_scaler.pkl

X_val shape : (7912, 48, 32)
y_val sepsis: 431 / 7912

Background : 100 non-sepsis patients
Explaining : 300 patients (100 sepsis + 200 non-sepsis)

Initialising KernelExplainer (this takes a moment)...
Computing SHAP values for 300 patients...
(This will take ~5-15 minutes — progress shown below)



100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [18:55<00:00,  3.78s/it]



✓ SHAP values computed. Shape: (300, 48, 32)
✓ Raw SHAP values saved → results/shap_values.npy

TOP 15 FEATURES BY SHAP IMPORTANCE
        feature  shap_value  rank
    num__ICULOS    0.000468     1
       num__WBC    0.000338     2
      num__Temp    0.000329     3
cat__Gender_0.0    0.000307     4
cat__Gender_1.0    0.000306     5
 num__Potassium    0.000287     6
     num__O2Sat    0.000278     7
 cat__Unit1_1.0    0.000277     8
       num__Hct    0.000276     9
       num__MAP    0.000275    10
   num__Glucose    0.000254    11
 cat__Unit1_0.0    0.000253    12
      num__Resp    0.000253    13
 num__Platelets    0.000252    14
       num__Hgb    0.000249    15

✓ Full table saved → results/feature_importance_shap.csv
✓ Bar chart saved → results/shap_bar_top15.png
✓ Beeswarm plot saved → results/shap_beeswarm.png
✓ Temporal heatmap saved → results/shap_temporal_heatmap.png
✓ Sepsis vs non-sepsis plot saved → results/shap_sepsis_vs_nonsepsis.png

SHAP ANALYSIS COMPLETE

Top 10 fea

In [4]:
# ── CRITICAL FIX — paste this in a new cell before running GradientExplainer ──

import torch.nn as nn

# cuDNN RNN backward pass only works in training mode
# We disable dropout manually so randomness doesn't affect SHAP values
shap_model.train()
for module in shap_model.modules():
    if isinstance(module, nn.Dropout):
        module.p = 0.0    # disable dropout without switching to eval()

print("✓ Model set to train() with dropout=0 — ready for GradientExplainer")

✓ Model set to train() with dropout=0 — ready for GradientExplainer


In [6]:
"""
Sepsis Prediction — SHAP GradientExplainer
============================================
Much faster than KernelExplainer (~30-60 seconds vs 5-15 minutes).
Uses backpropagation gradients — designed specifically for neural networks.

Run this in the same session as sepsis_lstm_v4.py so that
X_val, y_val, FEATURE_COLUMNS, best_overall_model are in memory.

Produces (all saved to results/):
  shap_bar_top15.png              ← top 15 features by mean |SHAP|   (paper figure)
  shap_beeswarm.png               ← direction of feature effects      (paper figure)
  shap_temporal_heatmap.png       ← which of 48 hours matter most     (paper figure)
  shap_sepsis_vs_nonsepsis.png    ← sepsis vs non-sepsis comparison
  shap_patient_waterfall.png      ← single patient explanation
  feature_importance_shap.csv     ← full ranked table                 (paper table)
  shap_values.npy                 ← raw values for reproducibility
"""

import os
import shap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH    = "saved_models/lstm_v4_final.pt"
SCALER_PATH   = "saved_models/lstm_v4_scaler.pkl"
RESULTS_DIR   = "results"
N_FEATURES    = 32
MAX_TIMESTEPS = 48
HIDDEN_DIM    = 64
NUM_LAYERS    = 2
DROPOUT       = 0.4

N_BACKGROUND  = 100   # background reference samples
N_EXPLAIN     = 500   # total patients to explain (fast enough with GradientExplainer)
N_EXPLAIN_SEP = 150   # of N_EXPLAIN, how many from sepsis class
RANDOM_SEED   = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Device — GradientExplainer runs on GPU unlike KernelExplainer
# ─────────────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Model definition (must match v4 exactly)
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,
        )
        lstm_out_dim = hidden_dim * 2
        self.attention  = nn.Linear(lstm_out_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        attn_scores  = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context)


# ─────────────────────────────────────────────────────────────────────────────
# 3. Load model and scaler
#    If best_overall_model is already in memory from v4, skip loading
# ─────────────────────────────────────────────────────────────────────────────
try:
    # Try to use model already in memory from v4 session
    shap_model = best_overall_model
    shap_model = shap_model.to(device)
    print("✓ Using best_overall_model from current session")
except NameError:
    # Standalone: load from disk
    shap_model = SepsisLSTM(
        input_dim  = N_FEATURES,
        hidden_dim = HIDDEN_DIM,
        num_layers = NUM_LAYERS,
        dropout    = DROPOUT,
    ).to(device)
    shap_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"✓ Model loaded from {MODEL_PATH}")

try:
    _ = FEATURE_COLUMNS
    _ = X_val
    _ = y_val
    print("✓ Using X_val, y_val, FEATURE_COLUMNS from current session")
except NameError:
    raise RuntimeError(
        "X_val, y_val, FEATURE_COLUMNS not found in memory.\n"
        "Run sepsis_lstm_v4.py first in the same session, or\n"
        "uncomment the standalone data loading block below."
    )

# ─────────────────────────────────────────────────────────────────────────────
# STANDALONE DATA LOADING (uncomment if running as separate script)
# ─────────────────────────────────────────────────────────────────────────────
# HEADER_FOLDER = "../processed_training_Ay"
# all_columns = set()
# for fname in os.listdir(HEADER_FOLDER):
#     if fname.endswith(".psv"):
#         with open(os.path.join(HEADER_FOLDER, fname)) as fh:
#             hdr = fh.readline().strip()
#             if hdr: all_columns.update(hdr.split("|"))
# FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")
#
# scaler = joblib.load(SCALER_PATH)
# # re-load and re-pad X_val, y_val from your data folders here

# ─────────────────────────────────────────────────────────────────────────────
# 4. Sample patients
#    Background: non-sepsis patients (what "normal" looks like)
#    Explain:    balanced mix of sepsis and non-sepsis
# ─────────────────────────────────────────────────────────────────────────────
sepsis_idx    = np.where(y_val == 1)[0]
nonsepsis_idx = np.where(y_val == 0)[0]

# Background — reference distribution for SHAP
bg_idx     = rng.choice(nonsepsis_idx, size=N_BACKGROUND, replace=False)
background = torch.tensor(X_val[bg_idx], dtype=torch.float32).to(device)

# Explanation set — balanced
n_sep       = min(N_EXPLAIN_SEP, len(sepsis_idx))
n_non       = N_EXPLAIN - n_sep
exp_sep_idx = rng.choice(sepsis_idx,    size=n_sep, replace=False)
exp_non_idx = rng.choice(nonsepsis_idx, size=n_non, replace=False)
exp_idx     = np.concatenate([exp_sep_idx, exp_non_idx])
exp_labels  = np.concatenate([np.ones(n_sep), np.zeros(n_non)]).astype(int)

to_explain  = torch.tensor(X_val[exp_idx], dtype=torch.float32).to(device)

print(f"\nBackground : {N_BACKGROUND} non-sepsis patients")
print(f"Explaining : {len(exp_idx)} patients ({n_sep} sepsis + {n_non} non-sepsis)")
print(f"Input shape: {to_explain.shape}  (patients, timesteps, features)")

# ─────────────────────────────────────────────────────────────────────────────
# 5. GradientExplainer
#    - Preserves 3D shape (patients, timesteps, features) — no flattening needed
#    - Uses integrated gradients via backprop — much faster than KernelExplainer
#    - ranked=False returns raw gradient-based SHAP values
# ─────────────────────────────────────────────────────────────────────────────
# ── CRITICAL FIX ─────────────────────────────────────────────────────────────
# GradientExplainer needs model.train() because cuDNN RNN backward pass
# only works in training mode. Dropout is disabled manually so it does
# not introduce randomness into the SHAP values.
shap_model.train()
for module in shap_model.modules():
    if isinstance(module, nn.Dropout):
        module.p = 0.0          # disable dropout without switching to eval()
print("✓ Model set to train() with dropout=0 (required for GradientExplainer)")

print("\nInitialising GradientExplainer...")
explainer = shap.GradientExplainer(shap_model, background)

print(f"Computing SHAP values for {len(exp_idx)} patients...")
# ranked_outputs=None returns SHAP for the single output neuron
shap_values = explainer.shap_values(to_explain, ranked_outputs=None)
# shap_values shape: (n_explain, timesteps, features)  ← 3D preserved!

shap_3d = np.array(shap_values)
if shap_3d.ndim == 4:
    shap_3d = shap_3d[0]   # some SHAP versions wrap in extra dimension
print(f"✓ SHAP values computed. Shape: {shap_3d.shape}")

# Save raw values
np.save(os.path.join(RESULTS_DIR, "shap_values.npy"),      shap_3d)
np.save(os.path.join(RESULTS_DIR, "shap_exp_labels.npy"),  exp_labels)
print(f"✓ Raw SHAP values saved → results/shap_values.npy")

# Restore model to eval mode now that SHAP is done
shap_model.eval()
print("✓ Model restored to eval() mode")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Feature importance — mean |SHAP| across patients and timesteps
# ─────────────────────────────────────────────────────────────────────────────
shap_per_feature = np.abs(shap_3d).mean(axis=(0, 1))   # (n_features,)

feature_importance = pd.DataFrame({
    "feature":    FEATURE_COLUMNS,
    "shap_value": shap_per_feature,
}).sort_values("shap_value", ascending=False).reset_index(drop=True)
feature_importance["rank"] = range(1, len(feature_importance) + 1)

print("\n" + "="*45)
print("TOP 15 FEATURES BY SHAP IMPORTANCE")
print("="*45)
print(feature_importance.head(15).to_string(index=False))

fi_path = os.path.join(RESULTS_DIR, "feature_importance_shap.csv")
feature_importance.to_csv(fi_path, index=False)
print(f"\n✓ Full table saved → {fi_path}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — Horizontal bar: top 15 features (main paper figure)
# ─────────────────────────────────────────────────────────────────────────────
top15 = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors  = ["#1A73E8"] * 5 + ["#5BA4F5"] * 5 + ["#A8C8FA"] * 5
ax.barh(
    top15["feature"].values[::-1],
    top15["shap_value"].values[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.5
)
ax.set_xlabel("Mean |SHAP value| — average impact on sepsis probability", fontsize=10)
ax.set_title(
    "Feature Importance — SHAP GradientExplainer\n"
    "Sepsis Prediction LSTM · AmsterdamUMCdb · 48-hour window",
    fontsize=11, fontweight="bold"
)
ax.grid(True, alpha=0.3, axis="x")
ax.spines[["top", "right"]].set_visible(False)
for i, (val, feat) in enumerate(zip(
        top15["shap_value"].values[::-1],
        top15["feature"].values[::-1])):
    ax.text(val + max(top15["shap_value"]) * 0.01,
            i, f"{val:.5f}", va="center", fontsize=8)

fig.tight_layout()
p1 = os.path.join(RESULTS_DIR, "shap_bar_top15.png")
fig.savefig(p1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\n✓ Bar chart → {p1}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — Beeswarm: direction of feature effects
#           Red dot = high feature value pushes toward sepsis
#           Blue dot = high feature value pushes away from sepsis
# ─────────────────────────────────────────────────────────────────────────────

# Collapse timesteps: mean SHAP per patient per feature
shap_2d = shap_3d.mean(axis=1)                    # (n_explain, n_features)
X_2d    = X_val[exp_idx].mean(axis=1)             # mean feature value per patient

shap_exp = shap.Explanation(
    values        = shap_2d,
    base_values   = np.zeros(len(exp_idx)),
    data          = X_2d,
    feature_names = FEATURE_COLUMNS,
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.title(
    "SHAP Beeswarm — Direction of Feature Effects\n"
    "Red = high value → higher sepsis risk  |  Blue = high value → lower risk",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
p2 = os.path.join(RESULTS_DIR, "shap_beeswarm.png")
plt.savefig(p2, dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Beeswarm → {p2}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — Temporal heatmap: which of the 48 hours matter most
#           Rows = top 10 features, Columns = hour 0..47
#           Colour = mean |SHAP| — darker = more important at that hour
# ─────────────────────────────────────────────────────────────────────────────
top10_names = feature_importance.head(10)["feature"].tolist()
top10_idx   = [FEATURE_COLUMNS.index(f) for f in top10_names]

# Mean |SHAP| per timestep per feature across all patients
temporal_importance = np.abs(shap_3d).mean(axis=0)           # (timesteps, features)
temporal_top10      = temporal_importance[:, top10_idx].T     # (10, timesteps)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(temporal_top10, aspect="auto", cmap="YlOrRd", interpolation="nearest")

ax.set_yticks(range(len(top10_names)))
ax.set_yticklabels(top10_names, fontsize=9)
ax.set_xlabel("Hour before sepsis onset (0 = oldest, 47 = most recent)", fontsize=10)
ax.set_title(
    "Temporal SHAP Heatmap — When Do Features Matter?\n"
    "Darker colour = higher impact on prediction at that hour",
    fontsize=11, fontweight="bold"
)
xtick_pos = list(range(0, MAX_TIMESTEPS, 6))
ax.set_xticks(xtick_pos)
ax.set_xticklabels([f"h-{MAX_TIMESTEPS - x}" for x in xtick_pos], fontsize=8)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean |SHAP value|", fontsize=9)

fig.tight_layout()
p3 = os.path.join(RESULTS_DIR, "shap_temporal_heatmap.png")
fig.savefig(p3, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Temporal heatmap → {p3}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 — Sepsis vs Non-Sepsis SHAP comparison
# ─────────────────────────────────────────────────────────────────────────────
shap_sep    = np.abs(shap_3d[exp_labels == 1]).mean(axis=(0, 1))
shap_nonsep = np.abs(shap_3d[exp_labels == 0]).mean(axis=(0, 1))

combined    = (shap_sep + shap_nonsep) / 2
top12_idx_c = np.argsort(combined)[::-1][:12]
top12_names = [FEATURE_COLUMNS[i] for i in top12_idx_c]

x     = np.arange(len(top12_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, shap_sep[top12_idx_c],    width,
       label=f"Sepsis (n={n_sep})",     color="#E8431A", alpha=0.85)
ax.bar(x + width/2, shap_nonsep[top12_idx_c], width,
       label=f"Non-sepsis (n={n_non})", color="#1A73E8", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top12_names, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Mean |SHAP value|", fontsize=10)
ax.set_title(
    "Feature Importance: Sepsis vs Non-Sepsis Patients\n"
    "Shows which features the model relies on differently for each group",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
p4 = os.path.join(RESULTS_DIR, "shap_sepsis_vs_nonsepsis.png")
fig.savefig(p4, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Sepsis vs non-sepsis → {p4}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 5 — Waterfall plot for a single high-risk sepsis patient
#           Shows exactly HOW the model arrived at its prediction
#           Great for the paper's case study / qualitative analysis section
# ─────────────────────────────────────────────────────────────────────────────

# Pick the sepsis patient with highest predicted probability
shap_model.eval()  # already restored above
with torch.no_grad():
    probs_all = torch.sigmoid(
        shap_model(to_explain).squeeze(1)
    ).cpu().numpy()

sepsis_mask      = exp_labels == 1
sepsis_probs     = probs_all[sepsis_mask]
highest_risk_idx = np.argmax(sepsis_probs)   # index within sepsis subset
full_idx         = np.where(sepsis_mask)[0][highest_risk_idx]

patient_shap = shap_3d[full_idx].mean(axis=0)  # mean over timesteps (n_features,)
patient_data = X_val[exp_idx[full_idx]].mean(axis=0)

patient_exp = shap.Explanation(
    values        = patient_shap,
    base_values   = float(explainer.expected_value)
                    if hasattr(explainer, "expected_value") else 0.0,
    data          = patient_data,
    feature_names = FEATURE_COLUMNS,
)

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(patient_exp, max_display=15, show=False)
plt.title(
    f"SHAP Waterfall — Highest-Risk Sepsis Patient\n"
    f"Predicted probability: {sepsis_probs[highest_risk_idx]:.3f}",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
p5 = os.path.join(RESULTS_DIR, "shap_patient_waterfall.png")
plt.savefig(p5, dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Waterfall plot → {p5}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Final summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("SHAP ANALYSIS COMPLETE")
print(f"{'='*55}")
print(f"\nTop 10 features driving sepsis prediction:")
for _, row in feature_importance.head(10).iterrows():
    print(f"  {int(row['rank']):2}. {row['feature']:<25} SHAP: {row['shap_value']:.6f}")

print(f"\nAll files saved to results/:")
print(f"  shap_bar_top15.png           ← Figure: feature importance (use in paper)")
print(f"  shap_beeswarm.png            ← Figure: direction of effects")
print(f"  shap_temporal_heatmap.png    ← Figure: temporal importance (novel)")
print(f"  shap_sepsis_vs_nonsepsis.png ← Figure: group comparison")
print(f"  shap_patient_waterfall.png   ← Figure: single patient case study")
print(f"  feature_importance_shap.csv  ← Table:  full ranked list")
print(f"  shap_values.npy              ← Data:   raw values")

print(f"\nMethods section text:")
print(f"  'SHAP GradientExplainer was applied to {len(exp_idx)} validation")
print(f"  patients ({n_sep} sepsis, {n_non} non-sepsis) using {N_BACKGROUND}")
print(f"  background samples to quantify feature contributions via")
print(f"  backpropagation-based attribution on the trained LSTM.'")

Using device: cuda
✓ Using best_overall_model from current session
✓ Using X_val, y_val, FEATURE_COLUMNS from current session

Background : 100 non-sepsis patients
Explaining : 500 patients (150 sepsis + 350 non-sepsis)
Input shape: torch.Size([500, 48, 32])  (patients, timesteps, features)
✓ Model set to train() with dropout=0 (required for GradientExplainer)

Initialising GradientExplainer...
Computing SHAP values for 500 patients...


RuntimeError: cudnn RNN backward can only be called in training mode

In [ ]:
import shap, torch.nn as nn, numpy as np

# ── Setup ─────────────────────────────────────────────────────────────────
shap_model_cpu = shap_model.cpu()
shap_model_cpu.train()
for module in shap_model_cpu.modules():
    if isinstance(module, nn.Dropout):
        module.p = 0.0

background_cpu = background.cpu()
to_explain_cpu = to_explain.cpu()

# ── Compute ───────────────────────────────────────────────────────────────
print("Initialising GradientExplainer on CPU...")
explainer = shap.GradientExplainer(shap_model_cpu, background_cpu)

print(f"Computing SHAP values for {len(exp_idx)} patients...")
shap_values = explainer.shap_values(to_explain_cpu, ranked_outputs=1)

# ── Fix shape ─────────────────────────────────────────────────────────────
shap_3d = np.array(shap_values[0])          # list → array: (500, 48, 32, 1)
if shap_3d.ndim == 4:
    shap_3d = shap_3d.squeeze(-1)           # → (500, 48, 32)

print(f"✓ SHAP values shape: {shap_3d.shape}")
# Expected: (500, 48, 32)  =  patients × timesteps × features

# ── Restore model to GPU ──────────────────────────────────────────────────
shap_model.cuda()
shap_model.eval()
print("✓ Model restored to GPU eval()")

# ── Save raw values ───────────────────────────────────────────────────────
np.save("results/shap_values.npy",     shap_3d)
np.save("results/shap_exp_labels.npy", exp_labels)
print("✓ Raw SHAP values saved")

Initialising GradientExplainer on CPU...
Computing SHAP values for 500 patients...


In [ ]:
# 6. Feature importance — mean |SHAP| across patients and timesteps
# ─────────────────────────────────────────────────────────────────────────────
shap_per_feature = np.abs(shap_3d).mean(axis=(0, 1))   # (n_features,)
 
feature_importance = pd.DataFrame({
    "feature":    FEATURE_COLUMNS,
    "shap_value": shap_per_feature,
}).sort_values("shap_value", ascending=False).reset_index(drop=True)
feature_importance["rank"] = range(1, len(feature_importance) + 1)
 
print("\n" + "="*45)
print("TOP 15 FEATURES BY SHAP IMPORTANCE")
print("="*45)
print(feature_importance.head(15).to_string(index=False))
 
fi_path = os.path.join(RESULTS_DIR, "feature_importance_shap.csv")
feature_importance.to_csv(fi_path, index=False)
print(f"\n✓ Full table saved → {fi_path}")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — Horizontal bar: top 15 features (main paper figure)
# ─────────────────────────────────────────────────────────────────────────────
top15 = feature_importance.head(15)
 
fig, ax = plt.subplots(figsize=(9, 6))
colors  = ["#1A73E8"] * 5 + ["#5BA4F5"] * 5 + ["#A8C8FA"] * 5
ax.barh(
    top15["feature"].values[::-1],
    top15["shap_value"].values[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.5
)
ax.set_xlabel("Mean |SHAP value| — average impact on sepsis probability", fontsize=10)
ax.set_title(
    "Feature Importance — SHAP GradientExplainer\n"
    "Sepsis Prediction LSTM · AmsterdamUMCdb · 48-hour window",
    fontsize=11, fontweight="bold"
)
ax.grid(True, alpha=0.3, axis="x")
ax.spines[["top", "right"]].set_visible(False)
for i, (val, feat) in enumerate(zip(
        top15["shap_value"].values[::-1],
        top15["feature"].values[::-1])):
    ax.text(val + max(top15["shap_value"]) * 0.01,
            i, f"{val:.5f}", va="center", fontsize=8)
 
fig.tight_layout()
p1 = os.path.join(RESULTS_DIR, "shap_bar_top15.png")
fig.savefig(p1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\n✓ Bar chart → {p1}")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — Beeswarm: direction of feature effects
#           Red dot = high feature value pushes toward sepsis
#           Blue dot = high feature value pushes away from sepsis
# ─────────────────────────────────────────────────────────────────────────────
 
# Collapse timesteps: mean SHAP per patient per feature
shap_2d = shap_3d.mean(axis=1)                    # (n_explain, n_features)
X_2d    = X_val[exp_idx].mean(axis=1)             # mean feature value per patient
 
shap_exp = shap.Explanation(
    values        = shap_2d,
    base_values   = np.zeros(len(exp_idx)),
    data          = X_2d,
    feature_names = FEATURE_COLUMNS,
)
 
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.title(
    "SHAP Beeswarm — Direction of Feature Effects\n"
    "Red = high value → higher sepsis risk  |  Blue = high value → lower risk",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
p2 = os.path.join(RESULTS_DIR, "shap_beeswarm.png")
plt.savefig(p2, dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Beeswarm → {p2}")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — Temporal heatmap: which of the 48 hours matter most
#           Rows = top 10 features, Columns = hour 0..47
#           Colour = mean |SHAP| — darker = more important at that hour
# ─────────────────────────────────────────────────────────────────────────────
top10_names = feature_importance.head(10)["feature"].tolist()
top10_idx   = [FEATURE_COLUMNS.index(f) for f in top10_names]
 
# Mean |SHAP| per timestep per feature across all patients
temporal_importance = np.abs(shap_3d).mean(axis=0)           # (timesteps, features)
temporal_top10      = temporal_importance[:, top10_idx].T     # (10, timesteps)
 
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(temporal_top10, aspect="auto", cmap="YlOrRd", interpolation="nearest")
 
ax.set_yticks(range(len(top10_names)))
ax.set_yticklabels(top10_names, fontsize=9)
ax.set_xlabel("Hour before sepsis onset (0 = oldest, 47 = most recent)", fontsize=10)
ax.set_title(
    "Temporal SHAP Heatmap — When Do Features Matter?\n"
    "Darker colour = higher impact on prediction at that hour",
    fontsize=11, fontweight="bold"
)
xtick_pos = list(range(0, MAX_TIMESTEPS, 6))
ax.set_xticks(xtick_pos)
ax.set_xticklabels([f"h-{MAX_TIMESTEPS - x}" for x in xtick_pos], fontsize=8)
 
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean |SHAP value|", fontsize=9)
 
fig.tight_layout()
p3 = os.path.join(RESULTS_DIR, "shap_temporal_heatmap.png")
fig.savefig(p3, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Temporal heatmap → {p3}")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 — Sepsis vs Non-Sepsis SHAP comparison
# ─────────────────────────────────────────────────────────────────────────────
shap_sep    = np.abs(shap_3d[exp_labels == 1]).mean(axis=(0, 1))
shap_nonsep = np.abs(shap_3d[exp_labels == 0]).mean(axis=(0, 1))
 
combined    = (shap_sep + shap_nonsep) / 2
top12_idx_c = np.argsort(combined)[::-1][:12]
top12_names = [FEATURE_COLUMNS[i] for i in top12_idx_c]
 
x     = np.arange(len(top12_names))
width = 0.35
 
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, shap_sep[top12_idx_c],    width,
       label=f"Sepsis (n={n_sep})",     color="#E8431A", alpha=0.85)
ax.bar(x + width/2, shap_nonsep[top12_idx_c], width,
       label=f"Non-sepsis (n={n_non})", color="#1A73E8", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top12_names, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Mean |SHAP value|", fontsize=10)
ax.set_title(
    "Feature Importance: Sepsis vs Non-Sepsis Patients\n"
    "Shows which features the model relies on differently for each group",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.spines[["top", "right"]].set_visible(False)
 
fig.tight_layout()
p4 = os.path.join(RESULTS_DIR, "shap_sepsis_vs_nonsepsis.png")
fig.savefig(p4, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Sepsis vs non-sepsis → {p4}")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 5 — Waterfall plot for a single high-risk sepsis patient
#           Shows exactly HOW the model arrived at its prediction
#           Great for the paper's case study / qualitative analysis section
# ─────────────────────────────────────────────────────────────────────────────
 
# Pick the sepsis patient with highest predicted probability
shap_model.eval()  # already restored above
with torch.no_grad():
    probs_all = torch.sigmoid(
        shap_model(to_explain).squeeze(1)
    ).cpu().numpy()
 
sepsis_mask      = exp_labels == 1
sepsis_probs     = probs_all[sepsis_mask]
highest_risk_idx = np.argmax(sepsis_probs)   # index within sepsis subset
full_idx         = np.where(sepsis_mask)[0][highest_risk_idx]
 
patient_shap = shap_3d[full_idx].mean(axis=0)  # mean over timesteps (n_features,)
patient_data = X_val[exp_idx[full_idx]].mean(axis=0)
 
patient_exp = shap.Explanation(
    values        = patient_shap,
    base_values   = float(explainer.expected_value)
                    if hasattr(explainer, "expected_value") else 0.0,
    data          = patient_data,
    feature_names = FEATURE_COLUMNS,
)
 
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(patient_exp, max_display=15, show=False)
plt.title(
    f"SHAP Waterfall — Highest-Risk Sepsis Patient\n"
    f"Predicted probability: {sepsis_probs[highest_risk_idx]:.3f}",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
p5 = os.path.join(RESULTS_DIR, "shap_patient_waterfall.png")
plt.savefig(p5, dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Waterfall plot → {p5}")
 
# ─────────────────────────────────────────────────────────────────────────────
# 7. Final summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("SHAP ANALYSIS COMPLETE")
print(f"{'='*55}")
print(f"\nTop 10 features driving sepsis prediction:")
for _, row in feature_importance.head(10).iterrows():
    print(f"  {int(row['rank']):2}. {row['feature']:<25} SHAP: {row['shap_value']:.6f}")
 
print(f"\nAll files saved to results/:")
print(f"  shap_bar_top15.png           ← Figure: feature importance (use in paper)")
print(f"  shap_beeswarm.png            ← Figure: direction of effects")
print(f"  shap_temporal_heatmap.png    ← Figure: temporal importance (novel)")
print(f"  shap_sepsis_vs_nonsepsis.png ← Figure: group comparison")
print(f"  shap_patient_waterfall.png   ← Figure: single patient case study")
print(f"  feature_importance_shap.csv  ← Table:  full ranked list")
print(f"  shap_values.npy              ← Data:   raw values")
 
print(f"\nMethods section text:")
print(f"  'SHAP GradientExplainer was applied to {len(exp_idx)} validation")
print(f"  patients ({n_sep} sepsis, {n_non} non-sepsis) using {N_BACKGROUND}")
print(f"  background samples to quantify feature contributions via")
print(f"  backpropagation-based attribution on the trained LSTM.'")